# Sequential model comparison

Run all cells in a fresh Colab CPU runtime. One queue trains linear regression, histogram gradient boosting, a bounded random forest, and an MLP **sequentially**, saving each model and its validation results to Drive before moving on. Ordinary model failures are recorded and do not stop later models.

Run notebook `02` first. This notebook restores its saved geometry partitions and checks the processed CSV's SHA256 fingerprint; it never generates a different split. By default it uses the latest baseline `run.json` in Drive.

Open the newly published notebook from GitHub, not an older saved Colab copy. Setup clones or fast-forwards a clean source checkout without discarding edits. All dataset processing and real training happen in Colab.

In [ ]:
from pathlib import Path
import subprocess
import sys

from google.colab import drive

PROJECT_ROOT = Path('/content/airfoil-ml-surrogate')
REPO_URL = 'https://github.com/gaurav-zz/ML-for-CFD.git'
if not PROJECT_ROOT.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_ROOT)], check=True)
if not (PROJECT_ROOT / '.git').exists():
    raise FileNotFoundError(f'Expected a Git checkout at {PROJECT_ROOT}. Use a fresh Colab runtime.')
if subprocess.check_output(['git', 'status', '--porcelain'], cwd=PROJECT_ROOT, text=True).strip():
    raise RuntimeError('The source checkout has local edits. Preserve them and use a fresh Colab runtime.')
previous_revision = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT_ROOT, text=True).strip()
subprocess.run(['git', 'pull', '--ff-only', 'origin', 'main'], cwd=PROJECT_ROOT, check=True)
SOURCE_REVISION = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT_ROOT, text=True).strip()
if SOURCE_REVISION != previous_revision and any(name == 'src' or name.startswith('src.') for name in sys.modules):
    raise RuntimeError('Source updated after project modules were imported. Restart the session, then run all cells again.')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_ROOT / 'requirements.txt')],
    check=True,
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'Source checkout: {PROJECT_ROOT} at {SOURCE_REVISION}')

## Restore the baseline experiment

Leave `BASELINE_RUN = None` to select the latest saved baseline automatically, or set it to a specific baseline `run.json` path. Keep the same processed CSV used for that baseline. A mismatch stops the run before any model training.

In [ ]:
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/airfoil-ml-surrogate')
PROCESSED_PATH = DRIVE_ROOT / 'data' / 'processed' / 'airfoil_learning.csv'
BASELINE_RUN = None

In [ ]:
from copy import deepcopy
from datetime import datetime, timezone
from importlib.metadata import version
import gc
import json

import joblib
import numpy as np
import pandas as pd

from src.training.comparison import (
    DEFAULT_MODEL_CONFIGS, compare_models, evaluate_selected_models, load_baseline_split,
)

if BASELINE_RUN is None:
    baseline_runs = sorted((DRIVE_ROOT / 'experiments' / 'baseline').glob('*/run.json'))
    if not baseline_runs:
        raise FileNotFoundError('No saved baseline run.json found in Drive. Complete notebook 02 first.')
    BASELINE_RUN = baseline_runs[-1]
else:
    BASELINE_RUN = Path(BASELINE_RUN)
print(f'Restoring baseline: {BASELINE_RUN}', flush=True)
split, feature_columns, target_columns, baseline_metadata = load_baseline_split(PROCESSED_PATH, BASELINE_RUN)

X_train = split.train[feature_columns]
y_train = split.train[target_columns]
X_validation = split.validation[feature_columns]
y_validation = split.validation[target_columns]
print(f'Features: {len(feature_columns)}; targets: {target_columns}')
for partition in ['train', 'validation', 'test']:
    frame = getattr(split, partition)
    print(f'{partition}: {len(frame):,} rows, {frame.geometry_id.nunique():,} geometries')
print('Dataset fingerprint and original geometry partitions verified.')

## Train the queue

The default queue uses the full, identical training and validation partitions. The forest caps its tree count, depth, leaf count, per-tree sampling, and worker count. The MLP uses a bounded architecture/iteration budget. No GPU is required by these scikit-learn models.

Nonlinear models standardize targets using training data only and return predictions in the original `Cl`/`Cd` units. This balances shared forest/MLP training losses. Gradient boosting and MLP use fixed iteration budgets rather than internal random row holdouts.

Full quadratic features are intentionally excluded: 103 inputs expand to 5,459 features, about 23.5 GB for one float64 training matrix at 538,380 rows, before copies or fitting.

Settings can be adjusted in `MODEL_CONFIGS` before training. Each run gets a new timestamped Drive directory. Rerunning the training cell starts a new queue; it does not overwrite or automatically resume an earlier run. Warnings and failures are recorded in `run.json` and each model's `result.json`.

In [ ]:
MODEL_CONFIGS = deepcopy(DEFAULT_MODEL_CONFIGS)
display(pd.DataFrame(MODEL_CONFIGS))

In [ ]:
run_started = datetime.now(timezone.utc)
RUN_OUTPUT = DRIVE_ROOT / 'experiments' / 'model_comparison' / run_started.strftime('%Y%m%dT%H%M%S%fZ')
run_metadata = {
    'created_at_utc': run_started.isoformat(),
    'git_commit': SOURCE_REVISION,
    'python_version': sys.version,
    'packages': {name: version(name) for name in ['numpy', 'pandas', 'scikit-learn', 'joblib', 'matplotlib']},
    'baseline_run': str(BASELINE_RUN),
    'baseline_git_commit': baseline_metadata['git_commit'],
    'dataset': baseline_metadata['dataset'],
    'feature_columns': feature_columns,
    'split_geometries': baseline_metadata['split_geometries'],
    'split_rows': baseline_metadata['split_rows'],
}
validation_results = compare_models(
    x_train=X_train, y_train=y_train,
    x_validation=X_validation, y_validation=y_validation,
    target_names=target_columns, output_dir=RUN_OUTPUT,
    model_configs=MODEL_CONFIGS, run_metadata=run_metadata,
)
print(f'Comparison artifacts: {RUN_OUTPUT}')

## Compare validation performance

Rank by validation RMSE separately for each target, not by a combined error across differently scaled coefficients. The best `Cl` model and best `Cd` model may differ. Timings are for the full fit/prediction batch, not separate per-target timings. Each model is saved before the next one starts.

In [ ]:
import matplotlib.pyplot as plt

comparison_report = json.loads((RUN_OUTPUT / 'run.json').read_text())
failed_models = [name for name, result in comparison_report['results'].items() if result['status'] == 'failed']
if failed_models:
    print(f'Models that failed: {failed_models}. See run.json for diagnostics.')
if validation_results.empty:
    raise RuntimeError(f'No models completed successfully. Inspect {RUN_OUTPUT / "run.json"}.')
display(validation_results.sort_values(['target', 'rmse', 'model']).set_index(['target', 'model']))
print('Validation-selected models:', comparison_report['selected_models'])

fig, axes = plt.subplots(1, len(target_columns), figsize=(6 * len(target_columns), 4), squeeze=False)
for index, target in enumerate(target_columns):
    scores = validation_results[validation_results.target == target].sort_values(['rmse', 'model'])
    axes[0, index].barh(scores['model'], scores['rmse'])
    axes[0, index].invert_yaxis()
    axes[0, index].set_title(f'{target}: validation RMSE')
    axes[0, index].set_xlabel('RMSE (original coefficient units)')
fig.tight_layout()
fig.savefig(RUN_OUTPUT / 'validation_rmse.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

## Final test report for the selected models

Only validation-selected winners are evaluated here. Selection is already frozen in `run.json`; test scores never rerank the queue. Use validation results, not this final report, for subsequent tuning. Test metrics use every held-out row; the scatter plots use a reproducible sample of at most 5,000 rows for readability.

In [ ]:
test_results = evaluate_selected_models(
    RUN_OUTPUT, split.test[feature_columns], split.test[target_columns], target_names=target_columns,
)
display(test_results.set_index(['target', 'model']))

In [ ]:
selected_models = comparison_report['selected_models']
plot_rows = split.test.sample(n=min(5000, len(split.test)), random_state=42)
fig, axes = plt.subplots(1, len(target_columns), figsize=(6 * len(target_columns), 5), squeeze=False)
for model_name in dict.fromkeys(selected_models.values()):
    selected_model = joblib.load(RUN_OUTPUT / model_name / f'{model_name}.joblib')
    predictions = np.asarray(selected_model.predict(plot_rows[feature_columns])).reshape(len(plot_rows), len(target_columns))
    for index, target in enumerate(target_columns):
        if selected_models[target] != model_name:
            continue
        actual = plot_rows[target].to_numpy()
        predicted = predictions[:, index]
        axis = axes[0, index]
        axis.scatter(actual, predicted, alpha=0.3, s=10)
        lower = min(actual.min(), predicted.min())
        upper = max(actual.max(), predicted.max())
        axis.plot([lower, upper], [lower, upper], '--', color='black')
        axis.set_title(f'{target}: {model_name}')
        axis.set_xlabel(f'Actual {target}')
        axis.set_ylabel(f'Predicted {target}')
        axis.grid(alpha=0.3)
    del selected_model, predictions
    gc.collect()
fig.suptitle(f'{len(plot_rows):,} sampled test rows; metrics use all {len(split.test):,} test rows')
fig.tight_layout()
fig.savefig(RUN_OUTPUT / 'selected_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)
print(f'All comparison results saved to: {RUN_OUTPUT}')